# Graph-Enhanced HyDE on SciFact

Pipeline:
1. **Extract ER** — LLM extracts entities + relationships from the query claim
2. **Graph search** — embed extracted entities → LanceDB lookup → 1-hop relationship traversal
3. **Hypothetical answer** — LLM writes a scientific passage using claim + graph context
4. **Dense retrieval** — embed hypothesis with BGE → FAISS search against corpus

**Kernel**: `.venv` (Python 3.13, graphrag 3.0.9)  
**Requires**: `GRAPHRAG_API_KEY` in `.env`

In [1]:
# Install retrieval deps if not present in this venv
import importlib, subprocess, sys
for pkg, import_name in [("sentence_transformers", "sentence_transformers"), ("faiss", "faiss")]:
    try:
        importlib.import_module(import_name)
    except ModuleNotFoundError:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               pkg if pkg != "faiss" else "faiss-cpu"])
print("Deps OK")

Installing sentence_transformers...
Deps OK



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os, sys, json
from pathlib import Path

# ── paths ────────────────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path(".").resolve()          # graphrag_quickstart/
REPO_ROOT    = NOTEBOOK_DIR.parent
OUTPUT_DIR   = NOTEBOOK_DIR / "output"
LANCEDB_URI  = str(OUTPUT_DIR / "lancedb")
RESULTS_DIR  = NOTEBOOK_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)

sys.path.insert(0, str(REPO_ROOT / "hydeOnSciFact"))

# ── load .env ────────────────────────────────────────────────────────────────
from dotenv import load_dotenv
load_dotenv(NOTEBOOK_DIR / ".env")
API_KEY = os.environ["GRAPHRAG_API_KEY"]

print("NOTEBOOK_DIR :", NOTEBOOK_DIR)
print("LANCEDB_URI  :", LANCEDB_URI)
print("API key set  :", bool(API_KEY))

NOTEBOOK_DIR : /Users/mj/Desktop/CMU/2026 Spring/11711 Advanced Natural Language Processing/ANLP-HW34/graphrag_quickstart
LANCEDB_URI  : /Users/mj/Desktop/CMU/2026 Spring/11711 Advanced Natural Language Processing/ANLP-HW34/graphrag_quickstart/output/lancedb
API key set  : True


## 1. Load SciFact data

In [3]:
from load_data import load_scifact_data

corpus, queries, qrels, data_source = load_scifact_data(split="test")

query_ids      = list(queries.keys())
corpus_doc_ids = list(corpus.keys())
corpus_texts   = [corpus[did] for did in corpus_doc_ids]

print(f"Source : {data_source}")
print(f"Corpus : {len(corpus_texts):,} docs  |  Queries: {len(query_ids):,}")

/Users/mj/Desktop/CMU/2026 Spring/11711 Advanced Natural Language Processing/ANLP-HW34/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Source : mteb/scifact (corpus:corpus/corpus, queries:queries/queries, qrels:default/test)
Corpus : 5,183 docs  |  Queries: 300


## 2. Load graph artefacts

In [4]:
import pandas as pd

entities_df      = pd.read_parquet(OUTPUT_DIR / "entities.parquet")
relationships_df = pd.read_parquet(OUTPUT_DIR / "relationships.parquet")

# Fast lookup: entity title → description
entity_desc: dict[str, str] = dict(zip(entities_df["title"], entities_df["description"]))

# Adjacency: entity title → list of (neighbour_title, relationship_description)
adjacency: dict[str, list[tuple[str, str]]] = {}
for _, row in relationships_df.iterrows():
    src, tgt, desc = row["source"], row["target"], row["description"]
    adjacency.setdefault(src, []).append((tgt, desc))
    adjacency.setdefault(tgt, []).append((src, desc))

print(f"Entities: {len(entities_df):,}  |  Relationships: {len(relationships_df):,}")

Entities: 25,791  |  Relationships: 35,372


## 3. Connect LanceDB for entity vector search

In [5]:
import lancedb
from openai import OpenAI

openai_client   = OpenAI(api_key=API_KEY)
EMBED_MODEL_OAI = "text-embedding-3-large"   # must match what built LanceDB

db            = lancedb.connect(LANCEDB_URI)
entity_table  = db.open_table("entity_description")

print("LanceDB table 'entity_description' opened.")
print("Row count:", entity_table.count_rows())

LanceDB table 'entity_description' opened.
Row count: 25791


## 4. Build corpus FAISS index (BGE)

We use the same BGE model as the baseline HyDE so that dense retrieval
performance is directly comparable.

In [7]:
from embed import Embedder
from retrieve import build_faiss_index

EMBED_MODEL_BGE = "BAAI/bge-base-en-v1.5"
embedder        = Embedder(model_name=EMBED_MODEL_BGE)

print("Encoding corpus with BGE...")
corpus_vectors = embedder.encode(corpus_texts)
faiss_index    = build_faiss_index(corpus_vectors)

print(f"FAISS index built: {faiss_index.ntotal:,} vectors, dim={corpus_vectors.shape[1]}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7992.67it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding corpus with BGE...
FAISS index built: 5,183 vectors, dim=768


## 5. Pipeline functions

### Step 1 — Extract entities & relationships from query
### Step 2 — Graph search (LanceDB + 1-hop traversal)
### Step 3 — Generate hypothetical answer with graph context
### Step 4 — Embed hypothesis → FAISS retrieval

In [8]:
import numpy as np

# ── Step 1: extract ER from query ────────────────────────────────────────────

def extract_er_from_query(claim: str) -> dict:
    """Use LLM to extract entities and relationships from a scientific claim."""
    prompt = (
        "Extract biomedical entities and relationships from the following scientific claim.\n"
        "Return JSON with two keys:\n"
        "  \"entities\": list of entity names (genes, proteins, diseases, drugs, "
        "biological processes, organisms, cell types, etc.)\n"
        "  \"relationships\": list of strings describing relations between entities\n"
        "Be concise. Use exact biomedical terminology.\n\n"
        f"Claim: {claim}\n\n"
        "JSON:"
    )
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=200,
        temperature=0,
        response_format={"type": "json_object"},
    )
    text = response.choices[0].message.content.strip()
    try:
        result = json.loads(text)
    except json.JSONDecodeError:
        result = {"entities": [], "relationships": []}
    return {
        "entities": result.get("entities", []),
        "relationships": result.get("relationships", []),
    }


# ── Step 2: graph search ──────────────────────────────────────────────────────

def _embed_texts_oai(texts: list[str]) -> list[list[float]]:
    """Batch embed with text-embedding-3-large."""
    if not texts:
        return []
    response = openai_client.embeddings.create(model=EMBED_MODEL_OAI, input=texts)
    return [item.embedding for item in response.data]


def search_graph(
    er: dict,
    top_k_entities: int = 3,
    max_hops: int = 1,
    max_rels_per_entity: int = 3,
) -> dict:
    """
    For each extracted entity, find the top-K similar entities in LanceDB,
    then collect their 1-hop relationships from the graph.
    Returns a dict with 'matched_entities' and 'relationships'.
    """
    query_entities = er.get("entities", [])
    if not query_entities:
        return {"matched_entities": [], "relationships": []}

    # Embed all query entities in one batch
    query_vectors = _embed_texts_oai(query_entities)

    matched_titles: list[str] = []
    seen_titles: set[str] = set()

    for vec in query_vectors:
        results = (
            entity_table
            .search(vec)
            .limit(top_k_entities)
            .to_pandas()
        )
        for _, row in results.iterrows():
            title = row.get("title", "")
            if title and title not in seen_titles:
                seen_titles.add(title)
                matched_titles.append(title)

    # Build entity context
    matched_entities = [
        {"title": t, "description": entity_desc.get(t, "")}
        for t in matched_titles
    ]

    # Collect 1-hop relationships for each matched entity
    seen_rels: set[str] = set()
    collected_rels: list[str] = []
    for title in matched_titles:
        for neighbour, rel_desc in adjacency.get(title, [])[:max_rels_per_entity]:
            rel_str = f"{title} — {rel_desc} — {neighbour}"
            if rel_str not in seen_rels:
                seen_rels.add(rel_str)
                collected_rels.append(rel_str)

    return {
        "matched_entities": matched_entities,
        "relationships": collected_rels,
    }


# ── Step 3: generate hypothetical answer ─────────────────────────────────────

GRAPH_HYDE_PROMPT = """\
You are a biomedical scientist. Using the scientific claim and the relevant
knowledge graph context below, write a concise scientific passage (2-4 sentences)
that a paper supporting or refuting this claim would contain.
Be specific, use precise terminology, and include relevant entities and mechanisms.

Claim: {claim}

Extracted entities from claim: {query_entities}
Extracted relationships from claim: {query_rels}

Related entities from knowledge graph:
{graph_entities}

Related relationships from knowledge graph:
{graph_rels}

Scientific passage:"""


def generate_graph_hyde(claim: str, er: dict, graph_context: dict) -> str:
    """Generate a hypothetical scientific passage enriched with graph context."""
    entity_lines = "\n".join(
        f"  - {e['title']}: {e['description'][:120]}"
        for e in graph_context["matched_entities"][:8]
    ) or "  (none found)"
    rel_lines = "\n".join(
        f"  - {r}"
        for r in graph_context["relationships"][:10]
    ) or "  (none found)"

    prompt = GRAPH_HYDE_PROMPT.format(
        claim=claim,
        query_entities=", ".join(er["entities"]) or "(none)",
        query_rels="; ".join(er["relationships"]) or "(none)",
        graph_entities=entity_lines,
        graph_rels=rel_lines,
    )

    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=256,
        temperature=0.3,
    )
    return (response.choices[0].message.content or "").strip()


print("Pipeline functions defined.")

Pipeline functions defined.


## 6. Run Graph-HyDE pipeline on all queries

In [ ]:
from tqdm import tqdm
from retrieve import retrieve_top_k

TOP_K  = 100
TOP_KS = [1, 3, 5, 10, 100]

per_query_rows: list[dict] = []
hypothetical_docs: list[str] = []

for qid in tqdm(query_ids, desc="Graph-HyDE"):
    claim = queries[qid]

    # Step 1 — extract ER from claim
    er = extract_er_from_query(claim)

    # Step 2 — search graph
    graph_ctx = search_graph(er)

    # Step 3 — generate hypothetical answer with graph context
    hyp_doc = generate_graph_hyde(claim, er, graph_ctx)
    hypothetical_docs.append(hyp_doc)

    per_query_rows.append({
        "query_id"           : qid,
        "query"              : claim,
        "extracted_entities" : er["entities"],
        "extracted_rels"     : er["relationships"],
        "graph_entities"     : [e["title"] for e in graph_ctx["matched_entities"]],
        "graph_rels"         : graph_ctx["relationships"],
        "hypothetical_doc"   : hyp_doc,
        "gold_doc_ids"       : sorted(qrels[qid]),
    })

print(f"\nGenerated {len(hypothetical_docs)} hypothetical passages.")

Graph-HyDE: 100%|██████████| 300/300 [20:43<00:00,  4.15s/it]


Generated 300 hypothetical passages.


In [ ]:
# Step 4 — embed hypothetical docs with BGE → FAISS retrieval
print("Encoding hypothetical docs with BGE...")
hyp_vectors = embedder.encode(hypothetical_docs)

_, all_indices = retrieve_top_k(faiss_index, hyp_vectors, TOP_K)

per_query_retrieved: dict[str, list[str]] = {}
for i, qid in enumerate(query_ids):
    retrieved_doc_ids = [corpus_doc_ids[j] for j in all_indices[i].tolist()]
    per_query_retrieved[qid] = retrieved_doc_ids
    per_query_rows[i]["retrieved_doc_ids"] = retrieved_doc_ids
    per_query_rows[i]["hit@1"]  = int(any(d in qrels[qid] for d in retrieved_doc_ids[:1]))
    per_query_rows[i]["hit@5"]  = int(any(d in qrels[qid] for d in retrieved_doc_ids[:5]))
    per_query_rows[i]["hit@10"] = int(any(d in qrels[qid] for d in retrieved_doc_ids[:10]))
    per_query_rows[i]["hit@100"] = int(any(d in qrels[qid] for d in retrieved_doc_ids[:100]))


print("Retrieval done.")

Encoding hypothetical docs with BGE...
Retrieval done.


## 7. Evaluate

In [ ]:
from evaluate import evaluate_run

metrics = evaluate_run(per_query_retrieved, qrels, TOP_KS)

print("\n=== Graph-HyDE Retrieval — SciFact ===\n")
for k in ["Recall@1", "Recall@5", "Recall@10", "Recall@100", "MRR@10", "nDCG@10"]:
    if k in metrics:
        print(f"  {k:<12}: {metrics[k]:.4f}")

print("\n--- Baseline comparison ---")
print("  BM25        Recall@10: 0.8073")
print("  BGE HyDE    Recall@10: 0.8902")


=== Graph-HyDE Retrieval — SciFact ===

  Recall@1    : 0.5483
  Recall@5    : 0.8028
  Recall@10   : 0.8701
  MRR@10      : 0.6751
  nDCG@10     : 0.7190

--- Baseline comparison ---
  BM25        Recall@10: 0.8073
  BGE HyDE    Recall@10: 0.8902


## 8. Save results

In [ ]:
metrics_payload = {
    "config": {
        "mode"              : "graph_hyde",
        "split"             : "test",
        "data_source"       : data_source,
        "llm_model"         : "gpt-4o-mini",
        "entity_embed_model": EMBED_MODEL_OAI,
        "retrieval_embed"   : EMBED_MODEL_BGE,
        "top_ks"            : TOP_KS,
    },
    "metrics": metrics,
}

metrics_path = RESULTS_DIR / "graph_hyde_metrics.json"
with metrics_path.open("w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2, ensure_ascii=False)

per_query_path = RESULTS_DIR / "graph_hyde_per_query.jsonl"
with per_query_path.open("w", encoding="utf-8") as f:
    for row in per_query_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"Saved metrics   : {metrics_path}")
print(f"Saved per-query : {per_query_path}")